# Chapter 2 — Measuring Local Sensitivity
## Foundations of Sensitivity Analysis — Arriola & Hyman (SIAM, 2026)

**Purpose:** Compute and visualize sensitivity indices for algebraic and SIR models.
Generate the tornado plot (Fig 2.1) and parameter table (Table 2.2).

**Key claims tested:**
- SI_k = SI_β = SI_τ = 1 for R₀ = kβτ (all exact)
- SI_L = 0 (lifespan absent from R₀)
- Reciprocal parameterization: SI_τ = −SI_γ
- Elasticity interpretation: SI = ∂ln(q)/∂ln(p)

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

FULL = False
print('Chapter 2: Measuring Local Sensitivity')

In [ ]:
# ── SIR Nominal Parameters (Table 2.2) ────────────────────────────────────────
def sir_nominal():
    """Return nominal SIR parameters as a dict.

    Returns
    -------
    dict with keys: c, beta, tau_R, tau_m, gamma_R, nu_m, N, S0, I0, R0_ic, R0, T
    """
    p = dict(
        c    = 5.0,       # contacts per person per day
        beta = 0.06,      # transmission probability per contact
        tau_R  = 7.0,       # mean infectious period (days)
        tau_m    = 10_000.0,  # mean lifespan (days)
        N    = 1_000,     # population size
        S0   = 999,       # initial susceptibles
        I0   = 1,         # initial infectious
        R0_ic= 0,         # initial recovered
        T    = 90,        # observation window (days)
    )
    p['gamma_R'] = 1.0 / p['tau_R']
    p['nu_m']    = 1.0 / p['tau_m']
    p['R0']    = p['c'] * p['beta'] * p['tau_R']
    return p

p = sir_nominal()
print(f"Nominal parameters:")
for key in ['c','beta','tau_R','tau_m','R0']:
    print(f"  {key:6s} = {p[key]}")

In [ ]:
# ── Verification Suite ─────────────────────────────────────────────────────────
print('=' * 60)
print('VERIFICATION SUITE')
print('=' * 60)

def sensitivity_index(q_func, p_val, dp=1e-7):
    """Compute normalized SI via centered finite difference."""
    q0  = q_func(p_val)
    dq  = (q_func(p_val*(1+dp)) - q_func(p_val*(1-dp))) / (2*dp*p_val)
    return (p_val / q0) * dq

# Analytic SIs for R₀ = kβτ
R0_fn = lambda pv, key: {
    'c':    lambda v: v*p['beta']*p['tau_R'],
    'beta': lambda v: p['c']*v*p['tau_R'],
    'tau_R':  lambda v: p['c']*p['beta']*v,
    'tau_m':    lambda v: p['c']*p['beta']*p['tau_R'],  # R0 independent of tau_m
}[key](pv)

expected = {'c': 1.0, 'beta': 1.0, 'tau_R': 1.0, 'tau_m': 0.0}
for param, exp_si in expected.items():
    pval = p[param]
    si = sensitivity_index(lambda v, pk=param: R0_fn(v, pk), pval)
    err = abs(si - exp_si)
    assert err < 1e-8, f'FAIL: SI_{param}={si:.8f}, expected={exp_si}'
    print(f'Test PASS: SI_{param}^R0 = {si:.8f}  (exact: {exp_si})')

# Reciprocal parameterization: SI_τ = -SI_γ
gamma_R_nom = p['gamma_R']
R0_gamma_R  = lambda g: p['c'] * p['beta'] / g
SI_gamma_R  = sensitivity_index(R0_gamma_R, gamma_R_nom)
SI_tau_R_v  = expected['tau_R']
assert abs(SI_gamma_R - (-SI_tau_R_v)) < 1e-8, 'FAIL: reciprocal rule'
print(f'Test PASS: SI_γ^R0 = {SI_gamma_R:.6f} = -SI_τ^R0 = {-SI_tau_R_v:.6f}')

# Elasticity form: SI = d(ln R0)/d(ln c)
R0_c = lambda c: c * p['beta'] * p['tau_R']
dk = 1e-7
d_lnR0_d_lnk = (np.log(R0_c(p['c']*(1+dk))) - np.log(R0_c(p['c']*(1-dk)))) / (2*dk)
assert abs(d_lnR0_d_lnk - 1.0) < 1e-8, 'FAIL: elasticity form'
print(f'Test PASS: d(lnR0)/d(lnk) = {d_lnR0_d_lnk:.8f}  (exact: 1)')

print(f'\nAll 6 verification tests PASSED.')
print('=' * 60)

In [ ]:
# ── Figure 1: Tornado Plot for R₀ (Fig 2.1 from book) ────────────────────────
params      = ['c', 'beta', 'tau_R', 'tau_m']       # identifier keys
params_disp = [r'$c$', r'$\beta$', r'$\tau_R$', r'$\tau_m$']  # display labels
si_vals = np.array([1.0, 1.0, 1.0, 0.0])
colors  = ['steelblue' if v > 0 else ('coral' if v < 0 else 'lightgray')
           for v in si_vals]

# Sort by absolute value (tornado ordering)
order = np.argsort(np.abs(si_vals))  # ascending → bottom of tornado
params_s  = [params[i] for i in order]
si_s      = si_vals[order]
colors_s  = [colors[i] for i in order]

fig, ax = plt.subplots(figsize=(7, 3.5))
y_pos = np.arange(len(params_s))
bars  = ax.barh(y_pos, si_s, color=colors_s, edgecolor='white', height=0.55)

for bar, val in zip(bars, si_s):
    xpos = bar.get_x() + bar.get_width() + 0.02 if val >= 0 else bar.get_x() - 0.02
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=10)

ax.set_yticks(y_pos)
disp_map = dict(zip(params, params_disp))
ax.set_yticklabels([disp_map[p] for p in params_s], fontsize=12)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel(r'Sensitivity index $\mathcal{S}_p^{\mathcal{R}_0}$', fontsize=12)
ax.set_title(r'Tornado plot: $\mathcal{R}_0 = c\beta\tau_R$  (Figure 2.1)', fontsize=11)
ax.set_xlim(-0.3, 1.5)
ax.grid(axis='x', alpha=0.3)

pos_patch = mpatches.Patch(color='steelblue', label='Positive SI (increasing)')
zero_patch = mpatches.Patch(color='lightgray', label='Zero SI (no effect)')
ax.legend(handles=[pos_patch, zero_patch], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('ch02_fig1_tornado.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch02_fig1_tornado.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tornado plot saved.')

In [ ]:
# ── Figure 2: SIR epidemic simulation at nominal parameters ───────────────────
def sir_rhs(t, y, c, beta, tau_R, tau_m, N):
    """SIR ODE right-hand side without demography."""
    S, I, R = y
    gamma_R = 1.0/tau_R
    dS = -c*beta*S*I/N
    dI =  c*beta*S*I/N - gamma_R*I
    dR =  gamma_R*I
    return [dS, dI, dR]

y0   = [p['S0'], p['I0'], p['R0_ic']]
tspan= [0, p['T']]
t_ev = np.linspace(0, p['T'], 500)

sol = solve_ivp(sir_rhs, tspan, y0,
                args=(p['c'], p['beta'], p['tau_R'], p['tau_m'], p['N']),
                dense_output=True, rtol=1e-10, atol=1e-12)
S_t, I_t, R_t = sol.sol(t_ev)

I_peak     = I_t.max()
t_peak     = t_ev[I_t.argmax()]
attack_rate= (p['S0'] - S_t[-1]) / p['N'] * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(t_ev, S_t, 'steelblue', lw=2, label='$S(t)$ Susceptible')
ax.plot(t_ev, I_t, 'coral',     lw=2, label='$I(t)$ Infectious')
ax.plot(t_ev, R_t, 'seagreen',  lw=2, label='$R(t)$ Recovered')
ax.axvline(t_peak, color='gray', ls=':', lw=1)
ax.annotate(f'Peak $I={I_peak:.0f}$\nat $t={t_peak:.0f}$ d',
            xy=(t_peak, I_peak), xytext=(t_peak+8, I_peak*0.85),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(fr'SIR epidemic  ($\mathcal{{R}}_0={p["R0"]:.1f}$, attack rate={attack_rate:.0f}%)', fontsize=11)
ax.legend(fontsize=10); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.semilogy(t_ev, I_t, 'coral', lw=2)
ax2.set_xlabel('Time (days)', fontsize=12)
ax2.set_ylabel('$I(t)$ (log scale)', fontsize=12)
ax2.set_title('Infectious compartment — log scale', fontsize=11)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('ch02_fig2_sir_epidemic.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch02_fig2_sir_epidemic.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'SIR simulation: peak I={I_peak:.1f} at t={t_peak:.1f} d, '
      f'attack rate={attack_rate:.1f}%')

In [ ]:
# ── Results & Download ─────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 2,
    'SI_R0': dict(zip(['c','beta','tau_R','tau_m'], si_vals.tolist())),
    'SIR_peak_I': float(I_peak), 'SIR_t_peak': float(t_peak),
    'SIR_attack_rate_pct': float(attack_rate),
    'figures': ['ch02_fig1_tornado.pdf', 'ch02_fig2_sir_epidemic.pdf']
}
print('Results:', RESULTS)

try:
    from google.colab import files
    for f in RESULTS['figures']:
        files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')